In [ ]:
import logging

import openeo.processes
from utils import urls, utils

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

# Parameters

In [ ]:
spatial_extent = {
    "west": 30.5503711040000994,
    "south": 1.0709279050000799,
    "east": 31.2229521229999989,
    "north": 1.5469373050000299,
}
temporal_extent = "2020-01-01"

In [ ]:
# very small test AOI
x = 30.7
y = 1.3
delta = 0.1
spatial_extent = {
    "west": x,
    "south": y - delta,
    "east": x + delta,
    "north": y,
}

In [ ]:
# minimum canopy cover to be considered forest
# should be one of (10, 20, 30, 40, 50, 60, 70, 80, 90)
# a value of 30 means > 30% canopy cover
canopy_cover_threshold = 30

# minimum likelihood to be considered natural forest
natural_forest_threshold = 0.08

# minimum connected area to be considered forest (m^2)
min_connected_area = 10000

# Script

In [ ]:
# collect outputs as we go, to built a multi-result process graph
process_graph_results = []

In [ ]:
# Natural Forests of the World 2020
# EPSG:32636 = UTM zone 36N
# dims: ['x', 'y', 'bands']
natural_forest = connection.load_stac(
    url=urls.NATURAL_FOREST_STAC,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["B0"],
)

In [ ]:
# remove the hidden time dimension
natural_forest = utils.drop_hidden_dimension(natural_forest, "t")

In [ ]:
# connection.describe_collection("ESA_WORLDCOVER_10M_2020_V1")

In [ ]:
# https://openeofed.dataspace.copernicus.eu/?discover=0&collection=ESA_WORLDCOVER_10M_2020_V1
# 10 m resolution
# EPSG:4326
# openEO backend: cdse
# ⚠️ on terrascope (and federated) backend this collection has different bands!
esa_worldcover = connection.load_collection(
    "ESA_WORLDCOVER_10M_2020_V1",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["MAP"],
)

In [ ]:
# remove the time dimension
esa_worldcover = esa_worldcover.drop_dimension("t")

In [ ]:
# connection.describe_collection("CLMS_TCD_PANTROPICAL_10M_YEARLY_V1")

In [ ]:
# tree cover density
# https://openeofed.dataspace.copernicus.eu/?discover=0&collection=CLMS_TCD_PANTROPICAL_10M_YEARLY_V1
# 10 m resolution
# EPSG:4326
# openEO backend: cdse
tree_cover_density = connection.load_collection(
    "CLMS_TCD_PANTROPICAL_10M_YEARLY_V1",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["map"],
)

In [ ]:
# remove the time dimension
tree_cover_density = tree_cover_density.drop_dimension("t")

In [ ]:
# align all 3 datasets in UTM zone 36N
esa_worldcover = esa_worldcover.resample_cube_spatial(natural_forest, method="near")
tree_cover_density = tree_cover_density.resample_cube_spatial(
    natural_forest, method="near"
)

In [ ]:
process_graph_results.append(
    natural_forest.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0000_natural_forest",
        },
    )
)
process_graph_results.append(
    esa_worldcover.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0000_esa_worldcover",
        },
    )
)
process_graph_results.append(
    tree_cover_density.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0000_tree_cover_density",
        },
    )
)

In [ ]:
condition_1 = natural_forest.band("B0") > natural_forest_threshold
condition_2 = esa_worldcover.band("MAP") == 10  # class 10 = tree cover
condition_3 = tree_cover_density.band("map") > canopy_cover_threshold
condition_4 = (
    # values above 100 = unclassifiable / no_data
    tree_cover_density.band("map") <= 100
)

In [ ]:
# condition_1 & condition_2 & condition_3 & condition_4
# BandMathException: 'Band math' between bands of different data cubes is not supported yet. 🙁

In [ ]:
# in order to do band math, we need to stack all of the inputs into a single DataCube
# https://github.com/Open-EO/openeo-python-client/issues/748

b1 = condition_1.add_dimension("bands", label="condition_1", type="bands")
b2 = condition_2.add_dimension("bands", label="condition_2", type="bands")
b3 = condition_3.add_dimension("bands", label="condition_3", type="bands")
b4 = condition_4.add_dimension("bands", label="condition_4", type="bands")

combined = b1.merge_cubes(b2).merge_cubes(b3).merge_cubes(b4)

In [ ]:
process_graph_results.append(
    combined.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0010_combined",
        },
    )
)
process_graph_results.append(
    combined.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_combined",
        },
    )
)
# check dtype of the merge_cubes bands
process_graph_results.append(
    combined.band("condition_1").save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_combined_condition_1",
        },
    )
)
process_graph_results.append(
    combined.band("condition_2").save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_combined_condition_2",
        },
    )
)
process_graph_results.append(
    combined.band("condition_3").save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_combined_condition_3",
        },
    )
)
process_graph_results.append(
    combined.band("condition_4").save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_combined_condition_4",
        },
    )
)

In [ ]:
# band math

forest_baseline = (
    combined.band("condition_1")
    & combined.band("condition_2")
    & combined.band("condition_3")
    & combined.band("condition_4")
)

# recreate the bands dimension
forest_baseline = forest_baseline.add_dimension("bands", label="B0", type="bands")

In [ ]:
process_graph_results.append(
    forest_baseline.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0020_forest_baseline",
        },
    )
)

In [ ]:
connectivity_udf = openeo.UDF.from_file(
    "../udf/connectivity_mask.py",
    runtime="Python",
    version="3.11",
    context={
        "pixel_area": 10 * 10,
        "min_connected_area": min_connected_area,
    },
)

In [ ]:
# mask where 1 = small region to be excluded
small_region_mask = forest_baseline.apply_neighborhood(
    connectivity_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    # overlap needs to be big enough the reasonably allow for min_pixels
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

In [ ]:
process_graph_results.append(
    small_region_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0030_small_region_mask",
        },
    )
)

In [ ]:
# apply_neighborhood UDF seems to return float32, even if it's a mask
# data types: https://github.com/locationtech/geotrellis/blob/master/raster/src/main/scala/geotrellis/raster/CellType.scala
small_region_mask = small_region_mask.convert_data_type("bool")

In [ ]:
process_graph_results.append(
    small_region_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0031_small_region_mask",
        },
    )
)

In [ ]:
# in order to do band math, we need to stack all of the inputs into a single DataCube
# https://github.com/Open-EO/openeo-python-client/issues/748

b1 = forest_baseline.rename_labels(dimension="bands", target=["forest_baseline"])
b2 = small_region_mask.rename_labels(dimension="bands", target=["small_region_mask"])

combined = b1.merge_cubes(b2)

In [ ]:
process_graph_results.append(
    combined.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0040_combined",
        },
    )
)
process_graph_results.append(
    combined.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0040_combined",
        },
    )
)
# check dtype of the merge_cubes bands - uint8
process_graph_results.append(
    combined.band("forest_baseline").save_result(
        format="GTiff",
        options={
            "filename_prefix": "0040_combined_forest_baseline",
        },
    )
)
process_graph_results.append(
    combined.band("small_region_mask").save_result(
        format="GTiff",
        options={
            "filename_prefix": "0040_combined_small_region_mask",
        },
    )
)

In [ ]:
# band math

forest_baseline = combined.band("forest_baseline") & utils.logical_not(
    combined.band("small_region_mask")
)

# recreate the bands dimension
forest_baseline = forest_baseline.add_dimension("bands", label="B0", type="bands")

In [ ]:
process_graph_results.append(
    forest_baseline.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0050_forest_baseline",
        },
    )
)

# Run job

In [ ]:
multi_result = openeo.MultiResult(process_graph_results)
job = multi_result.create_job()

In [ ]:
job.start_and_wait()

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-script/
!rm -r output-script/

In [ ]:
results.download_files("output-script/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)